In [4]:
# 1、引入Peft
from peft import LoraConfig
# 2、构建一个config对象
peft_config = LoraConfig(
    r  = 8,
    lora_alpha= 8,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM"
)

In [5]:
# 1、引入sft trainer
from trl.trainer.sft_trainer import SFTTrainer
from trl.trainer.sft_config import SFTConfig
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "logs/Qwen3-0.6B-SFT"
# 2、加载数据集
from datasets import load_dataset
dataset = load_dataset("json",data_files={"train":"data/keywords_data_train.jsonl","test":"data/keywords_data_test.jsonl"})
# 3、定义一个map函数，将数据集中的每个样本，转换成SFTTrainer支持的格式
def map_to_sft_format(example):
    conversation = example["conversation"]
    message_list = []
    for conv in conversation:
        for key ,value in conv.items():
            key = "user" if key == "human" else key
            message_list.append({"role":key,"content":value})

    return {"messages":message_list}
# 4、对dataset 进行转换
remove_list = list(dataset["train"][0].keys())
dataset = dataset.map(map_to_sft_format,batched=False,remove_columns=remove_list)
# 5、构造SFTConfig对象
config = SFTConfig(
    output_dir = "finetuned/Qwen3-0.6B-trl-sft",
    per_device_train_batch_size = 3,
    gradient_accumulation_steps = 4,
    learning_rate = 2e-5,
    max_steps = 3000,
    # 日志
    logging_steps = 100,
    
    report_to = ["tensorboard"],
    # 显存优化相关:
    bf16=True, # 混合精度
    gradient_checkpointing=True, # 梯度检查点
    activation_offloading = True, # CPU 卸载
    # 保存相关
    save_strategy = "steps",
    save_steps = 300,
    # 评估相关
    eval_steps = 300,
    eval_strategy = "steps",
    metric_for_best_model = "eval_loss",
    load_best_model_at_end=True,
    greater_is_better = False,
    max_length = 2500,
    
    chat_template_path="HuggingFaceTB/SmolLM3-3B",
    assistant_only_loss=True
)

In [ ]:
# 6、构造SFTTrainer对象
from transformers import AutoModelForCausalLM,AutoTokenizer
model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B-Base")
tokenizer = AutoTokenizer.from_pretrained("model/Qwen3-0.6B-Base")

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    args = config,
    train_dataset= dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config
)

trainer.train()
# 此时保存的就是AB矩阵，而不是原模型的所有的参数
trainer.save_model("finetuned/Qwen3-0.6B-SFT-Lora-Adapter")

In [ ]:
# 后面在推理阶段，需要将原模型的参数和适配器的参数合并起来
# 1、引入PeftModel
from peft import PeftModel
base_model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B-Base")
peft_model = PeftModel.from_pretrained(base_model,"finetuned/Qwen3-0.6B-SFT-Lora-Adapter")

# 2、将原模型的参数和适配器的参数合并起来
peft_model.merge_and_unload()
peft_model.save_pretrained("finetuned/Qwen3-0.6B-SFT-Lora-Merged")
